# Telco Customer Churn Prediction - End-to-End ML Pipeline

This notebook builds, trains, and evaluates complete Scikit-Learn pipelines using custom transformers:
- **`clean_cls`**: Automated data cleaning, type conversion, binary encoding, and dummy variables.
- **`CorrelationThresholdFilter`**: Feature selection based on Pearson correlation with the target.
- **`DecisionTreeClassifier`** & **`LogisticRegression`**: Classification models.

In [1]:
import sys
import os

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.tree import DecisionTreeClassifier, plot_tree
from sklearn.metrics import classification_report, confusion_matrix, roc_auc_score, roc_curve

sys.path.append(os.path.abspath("../dataset and other libs"))
from cleaningcls import clean_cls
from ctf import CorrelationThresholdFilter


## 1. Load Data & Train-Test Split
We pass raw features into the pipeline; the custom cleaning transformer handles all data preprocessing automatically.

In [ ]:
# Load raw dataset
df = pd.read_csv('D:/Repos/Chrun-Pridictor/dataset/WA_Fn-UseC_-Telco-Customer-Churn.csv')

# Separate features (X) and target (y)
X = df.drop(columns=['Churn'])
y = df['Churn'].map({'Yes': 1, 'No': 0})

# Stratified Train-Test Split
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print(f"Training set shape: {X_train.shape}")
print(f"Test set shape: {X_test.shape}")
print("\nTarget distribution in train set:")
print(y_train.value_counts(normalize=True).round(3))

FileNotFoundError: [Errno 2] No such file or directory: 'D:/Repos/Chrun-Pridictor/dataset/WA_Fn-UseC_-Telco-Customer-Churn.csv'

: 

## 2. Decision Tree Pipeline
A pipeline containing `clean_cls`, `CorrelationThresholdFilter`, and `DecisionTreeClassifier`.

In [ ]:
pipeline_dt = Pipeline([
    ('clean', clean_cls()),
    ('ctf', CorrelationThresholdFilter(threshold=0.23)),
    ('model', DecisionTreeClassifier(criterion='entropy',class_weight='balanced', max_depth=4, random_state=42))
])

# Fit pipeline on training data
pipeline_dt.fit(X_train, y_train)

In [ ]:
# Predictions & Evaluation
y_prob_dt = pipeline_dt.predict_proba(X_test)[:, 1]
y_pred_dt = pipeline_dt.predict(X_test)
# y_pred_dt = (pipeline_dt.predict_proba(X_test)[: , 1] > 0.45).astype(int)

auc_dt = roc_auc_score(y_test, y_prob_dt)
print('=== Decision Tree Pipeline Evaluation ===')
print(f'ROC-AUC Score: {auc_dt:.4f}')
print('\nConfusion Matrix:')
print(confusion_matrix(y_test, y_pred_dt))
print('\nClassification Report:')
print(classification_report(y_test, y_pred_dt))


In [ ]:
import pandas as pd

# 1. Get the trained model from your pipeline
model = pipeline_dt['model']

# 2. Get the feature importances (scores of how much the tree uses each feature)
importances = model.feature_importances_

# 3. Get the feature names from your pipeline's preprocessing steps

feature_names = pipeline_dt[:-1].get_feature_names_out()


# 4. Combine them into a clean DataFrame and sort by most important
feature_importance_df = pd.DataFrame({
    'Feature': feature_names,
    'Importance': importances
}).sort_values(by='Importance', ascending=False)

# important_f = feature_importance_df[feature_importance_df['Importance'] > 0]
active_features = feature_importance_df

print("=== Features Used by the Decision Tree (Ranked by Importance) ===")
print(active_features.to_string(index=False))

In [ ]:
feature_names

In [ ]:

sample_output = pipeline_dt['clean'].transform(X_train.head(5))
sample_output = pipeline_dt['ctf'].transform(sample_output)
feature_names = list(sample_output.columns)

In [ ]:
feature_names

In [ ]:
from trace_path import trace_customer_path
import trace_path

X_transformed = pipeline_dt['clean'].transform(X_train.head(1))
X_filtered = pipeline_dt['ctf'].transform(X_transformed)
trace_customer_path(pipeline_dt, X_filtered )